<a href="https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1
The paper reports that the proposed model improves prediction performance.

Methodology question:
How were the training and test sets separated? Was a time-aware or grouped split used to avoid data leakage?

## Finding 2
The paper reports that feature importance identifies the most useful signals.

Methodology question:
Were the important features measured only from information available before prediction, or could future information have leaked into the model?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Split

In Week 5 I used a random train/test split.

For this audit I use a time-aware split based on report_date. Earlier dates are used for training and later dates are used for testing. This better reflects how the model would be used in practice.

In [2]:

import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# Load Hugging Face token
token = userdata.get("HF_TOKEN")

# Load dataset
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=token,
    streaming=True
)

# Take first 1000 rows
df = pd.DataFrame(dataset.take(1000))

print(df.shape)
df.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(1000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [3]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Sort by date
df = df.sort_values("report_date")

# Recreate baseline score
df["baseline_score"] = (
    0.5 * df["gsc_impressions"] +
    0.3 * df["gsc_clicks"] +
    0.2 * df["sessions_organic"]
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "sessions_organic",
    "ga4_pageviews"
]

X = df[features]
y = df["baseline_score"]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R²:", r2_score(y_test, pred))

MAE: 0.14515999999999926
R²: 0.9878920532978279


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The model uses only information available at prediction time:
- gsc_impressions
- gsc_clicks
- sessions_organic
- ga4_pageviews

No future information or target variables were used.

No client identifiers or private information were included in the model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

Original Claim:
The model accurately predicts the best content to improve.

Rewritten Claim:
The model showed good performance on the observed test data. These results should be used as decision support and may not generalize to all future data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

 Two research findings discussed
 Methodology questions included
Honest time-aware split used
Leakage audit completed
 Claims rewritten using cautious language
Notebook executed successfully